In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------
# Parameters
# ------------------------
file_path = 'ERNE_TEG_cleaned.xlsx'
teg_metrics = ['R_time', 'K_time', 'Angle', 'MA', 'G', 'LY30']

AXIS_TICK_SIZE = 14
TITLE_SIZE = 18
LABEL_SIZE = 16

# Marker symbols and colors
marker_info = {
    "No Occlusion": {"marker": "o", "color": "black"},
    "Partial Occlusion": {"marker": "^", "color": "blue"},
    "Full Occlusion": {"marker": "s", "color": "red"}
}
legend_order = ["No Occlusion", "Partial Occlusion", "Full Occlusion"]

# Output directory
output_dir = 'TEG_By_Hemorrhage_Level'
os.makedirs(output_dir, exist_ok=True)

# ------------------------
# Load and preprocess data
# ------------------------
df = pd.read_excel(file_path)

time_col = 'Time Point'
group_col = 'Occlusion Group'
level_col = 'Hemorrhage Level'

# Ensure numeric time points
df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
df = df.dropna(subset=[time_col])
df[time_col] = df[time_col].astype(float)

# Fix naming
df[group_col] = df[group_col].replace({"None Occlusion": "No Occlusion"})

# ------------------------
# Plotting function
# ------------------------
def plot_teg_panels():
    hemorrhage_levels = sorted(df[level_col].unique())

    for metric in teg_metrics:
        fig, axes = plt.subplots(1, len(hemorrhage_levels), figsize=(18, 6), sharey=True)
        if len(hemorrhage_levels) == 1:
            axes = [axes]

        handles, labels = [], []

        for ax, hemorrhage_level in zip(axes, hemorrhage_levels):
            level_df = df[df[level_col] == hemorrhage_level]

            # Group by Occlusion Group & Time Point
            grouped_df = level_df.groupby([group_col, time_col])[metric].agg(['mean', 'std']).reset_index()

            for key in legend_order:  # enforce legend order
                subset = grouped_df[grouped_df[group_col] == key]
                if subset.empty:
                    continue

                line, = ax.plot(
                    subset[time_col],
                    subset['mean'],
                    marker=marker_info[key]["marker"],
                    markersize=8,
                    linewidth=2,
                    color=marker_info[key]["color"],
                    label=key
                )
                if key not in labels:
                    handles.append(line)
                    labels.append(key)

                # Error bands
                ax.fill_between(
                    subset[time_col],
                    subset['mean'] - subset['std'],
                    subset['mean'] + subset['std'],
                    alpha=0.2,
                    color=marker_info[key]["color"]
                )

            # Reference line at y=1
            ax.axhline(y=1, color='gray', linestyle='--', linewidth=1)

            ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=TITLE_SIZE)
            ax.set_xlabel("Time", fontsize=LABEL_SIZE)
            ax.grid(True, which='both', linestyle='--', linewidth=0.5)
            ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

        axes[0].set_ylabel(metric, fontsize=LABEL_SIZE)

        # Legend next to right-most panel
        axes[-1].legend(
            handles, labels,
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            fontsize=12,
            borderaxespad=0
        )

        plt.tight_layout()
        png_file = os.path.join(output_dir, f'TEG_{metric}_Panels.png')
        plt.savefig(png_file, dpi=300, bbox_inches='tight')
        plt.close()

# ------------------------
# Run plotting
# ------------------------
plot_teg_panels()


In [2]:
import os
import pandas as pd
import plotly.graph_objects as go

# Load data
file_path = 'ERNE_TEG_cleaned.xlsx'
df = pd.read_excel(file_path)

# Define TEG metrics
teg_metrics = [
'R_time', 
'K_time',
'Angle',
'MA', 
'G', 
'LY30'
]

# Column names
time_col = 'Time Point'  # Adjust if needed
group_col = 'Occlusion Group'
level_col = 'Hemorrhage Level'

# ST DEV

In [3]:
# Ensure numeric
df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
df = df.dropna(subset=[time_col])
df[time_col] = df[time_col].astype(float)

# Test group is occlusion group
df['Test Group'] = df[group_col]

# Output directory
output_dir = 'TEG_By_Hemorrhage_Level'
os.makedirs(output_dir, exist_ok=True)

# Marker and color setup
color_palette = ['black', 'blue', 'red', 'purple', 'orange', 'cyan']
occlusion_markers = {
    'Full Occlusion': 'square',
    'Partial Occlusion': 'triangle-up',
    'No Occlusion': 'circle'
}

# Plot by hemorrhage level
for hemorrhage_level in sorted(df[level_col].unique()):
    level_df = df[df[level_col] == hemorrhage_level]

    for metric in teg_metrics:
        fig = go.Figure()

        for i, test_group in enumerate(level_df['Test Group'].unique()):
            group_df = level_df[level_df['Test Group'] == test_group]

            grouped = group_df.groupby(time_col)[metric].agg(['mean', 'std']).reset_index()

            fig.add_trace(go.Scatter(
                x=grouped[time_col],
                y=grouped['mean'],
                mode='lines+markers',
                name=test_group,
                marker=dict(
                    symbol=occlusion_markers.get(test_group, 'circle'),
                    size=10,
                    color=color_palette[i % len(color_palette)]
                ),
                error_y=dict(type='data', array=grouped['std'], visible=True)
            ))

        fig.update_layout(
            title=f'{metric} Over Time (Hemorrhage Level: {hemorrhage_level}%)',
            title_font=dict(size=22),
            xaxis_title='Time',
            yaxis_title=metric,
#             xaxis=dict(
#                 showgrid=True,
#                 gridcolor='lightgrey',
#                 tickangle=45,
# #                 tickformat='.1f',  # Show one decimal if needed
#                 automargin=True
#             ),
            
            xaxis=dict(
                showgrid=True,
                gridcolor='lightgrey',
                tickangle=45,
                tickmode='array',
                tickvals=grouped[time_col].tolist(),  # <-- match x-ticks to actual values
                automargin=True
            ),

            yaxis=dict(
                showgrid=True,
                gridcolor='lightgrey'
            ),
            legend=dict(font=dict(size=12)),
            autosize=False,
            width=1000,
            height=600
        )

        # Save
        filename_base = f'TEG_{metric}_HemorrhageLevel{hemorrhage_level}'
        fig.write_image(os.path.join(output_dir, filename_base + '.png'))
        fig.write_html(os.path.join(output_dir, filename_base + '.html'))


# SEM

In [4]:
import os
import pandas as pd
import plotly.graph_objects as go

# Ensure numeric
df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
df = df.dropna(subset=[time_col])
df[time_col] = df[time_col].astype(float)

# Test group is occlusion group
df['Test Group'] = df[group_col]

# Output directory
output_dir = 'TEG_By_Hemorrhage_Level-SEM'
os.makedirs(output_dir, exist_ok=True)

# Marker and color setup
color_palette = ['black', 'blue', 'red', 'purple', 'orange', 'cyan']
occlusion_markers = {
    'Full Occlusion': 'square',
    'Partial Occlusion': 'triangle-up',
    'No Occlusion': 'circle'
}

# Plot by hemorrhage level
for hemorrhage_level in sorted(df[level_col].unique()):
    level_df = df[df[level_col] == hemorrhage_level]

    for metric in teg_metrics:
        fig = go.Figure()

        for i, test_group in enumerate(level_df['Test Group'].unique()):
            group_df = level_df[level_df['Test Group'] == test_group]

            # Compute mean, std, count, and SEM
            grouped_stats = group_df.groupby(time_col)[metric].agg(['mean', 'std', 'count']).reset_index()
            grouped_stats['sem'] = grouped_stats['std'] / grouped_stats['count'].pow(0.5)

            fig.add_trace(go.Scatter(
                x=grouped_stats[time_col],
                y=grouped_stats['mean'],
                mode='lines+markers',
                name=test_group,
                marker=dict(
                    symbol=occlusion_markers.get(test_group, 'circle'),
                    size=10,
                    color=color_palette[i % len(color_palette)]
                ),
                error_y=dict(type='data', array=grouped_stats['sem'], visible=True)
            ))

        fig.update_layout(
            title=f'{metric} Over Time (Hemorrhage Level: {hemorrhage_level}%)',
            title_font=dict(size=22),
            xaxis_title='Time',
            yaxis_title=metric,
            xaxis=dict(
                showgrid=True,
                gridcolor='lightgrey',
                tickangle=45,
                tickmode='array',
                tickvals=grouped_stats[time_col].tolist(),
                automargin=True
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor='lightgrey'
            ),
            legend=dict(font=dict(size=12)),
            autosize=False,
            width=1000,
            height=600
        )

        # Save
        filename_base = f'TEG_{metric}_HemorrhageLevel{hemorrhage_level}'
        fig.write_image(os.path.join(output_dir, filename_base + '.png'))
        fig.write_html(os.path.join(output_dir, filename_base + '.html'))


In [5]:
print('Done')

Done


# ST DEV

In [6]:
import os
import pandas as pd
import plotly.graph_objects as go



# Column references
time_col = 'Time Point'
level_col = 'Hemorrhage Level'
treatment_col = 'Occlusion Group'

# Ensure time is numeric
df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
df = df.dropna(subset=[time_col])
df[time_col] = df[time_col].astype(float)

# Combine Hemorrhage Level and Occlusion Group
df['Group Label'] = df[level_col].astype(str) + '% - ' + df[treatment_col]

# Output directory
output_dir = 'TEG_Combined_Hemorrhage_Treatment'
os.makedirs(output_dir, exist_ok=True)

# Style settings
color_palette = [
    'black', 'blue', 'red', 'purple', 'orange',
    'green', 'cyan', 'magenta', 'brown', 'pink', 'grey'
]
marker_symbols = [
    'circle', 'square', 'diamond', 'cross', 'x',
    'triangle-up', 'star', 'triangle-down', 'pentagon', 'hexagon'
]

# Generate plots
for metric in teg_metrics:
    fig = go.Figure()

    for i, group_label in enumerate(sorted(df['Group Label'].unique())):
        group_df = df[df['Group Label'] == group_label]

        # Group by time point
        grouped = group_df.groupby(time_col)[metric].agg(['mean', 'std']).reset_index()

        fig.add_trace(go.Scatter(
            x=grouped[time_col],
            y=grouped['mean'],
            mode='lines+markers',
            name=group_label,
            marker=dict(
                symbol=marker_symbols[i % len(marker_symbols)],
                size=10,
                color=color_palette[i % len(color_palette)]
            ),
            error_y=dict(
                type='data',
                array=grouped['std'],
                visible=True
            )
        ))

    fig.update_layout(
        title=f'{metric} Over Time (Grouped by Hemorrhage Level and Occlusion Group)',
        title_font=dict(size=22),
        xaxis_title='Time Point',
        yaxis_title=metric,
#         xaxis=dict(
#             showgrid=True,
#             gridcolor='lightgrey',
#             tickangle=45,
# #             tickformat='.1f'
#         ),

            
            xaxis=dict(
                showgrid=True,
                gridcolor='lightgrey',
                tickangle=45,
                tickmode='array',
                tickvals=grouped[time_col].tolist(),  # <-- match x-ticks to actual values
                automargin=True
            ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgrey'
        ),
        legend=dict(font=dict(size=10)),
        autosize=False,
        width=1200,
        height=700
    )

    # Save
    filename_base = f'TEG_{metric}'
    fig.write_image(os.path.join(output_dir, filename_base + '.png'))
    fig.write_html(os.path.join(output_dir, filename_base + '.html'))


# SEM

In [7]:
import os
import pandas as pd
import plotly.graph_objects as go

# Column references
time_col = 'Time Point'
level_col = 'Hemorrhage Level'
treatment_col = 'Occlusion Group'

# Ensure time is numeric
df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
df = df.dropna(subset=[time_col])
df[time_col] = df[time_col].astype(float)

# Combine Hemorrhage Level and Occlusion Group
df['Group Label'] = df[level_col].astype(str) + '% - ' + df[treatment_col]

# Output directory
output_dir = 'TEG_Combined_Hemorrhage_Treatment-SEM'
os.makedirs(output_dir, exist_ok=True)

# Style settings
color_palette = [
    'black', 'blue', 'red', 'purple', 'orange',
    'green', 'cyan', 'magenta', 'brown', 'pink', 'grey'
]
marker_symbols = [
    'circle', 'square', 'diamond', 'cross', 'x',
    'triangle-up', 'star', 'triangle-down', 'pentagon', 'hexagon'
]

# Generate plots
for metric in teg_metrics:
    fig = go.Figure()

    for i, group_label in enumerate(sorted(df['Group Label'].unique())):
        group_df = df[df['Group Label'] == group_label]

        # Group by time point and compute SEM
        grouped_stats = group_df.groupby(time_col)[metric].agg(['mean', 'std', 'count']).reset_index()
        grouped_stats['sem'] = grouped_stats['std'] / grouped_stats['count'].pow(0.5)

        fig.add_trace(go.Scatter(
            x=grouped_stats[time_col],
            y=grouped_stats['mean'],
            mode='lines+markers',
            name=group_label,
            marker=dict(
                symbol=marker_symbols[i % len(marker_symbols)],
                size=10,
                color=color_palette[i % len(color_palette)]
            ),
            error_y=dict(
                type='data',
                array=grouped_stats['sem'],
                visible=True
            )
        ))

    fig.update_layout(
        title=f'{metric} Over Time (Grouped by Hemorrhage Level and Occlusion Group)',
        title_font=dict(size=22),
        xaxis_title='Time Point',
        yaxis_title=metric,
        xaxis=dict(
            showgrid=True,
            gridcolor='lightgrey',
            tickangle=45,
            tickmode='array',
            tickvals=grouped_stats[time_col].tolist(),
            automargin=True
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgrey'
        ),
        legend=dict(font=dict(size=10)),
        autosize=False,
        width=1200,
        height=700
    )

    # Save
    filename_base = f'TEG_{metric}'
    fig.write_image(os.path.join(output_dir, filename_base + '.png'))
    fig.write_html(os.path.join(output_dir, filename_base + '.html'))


In [8]:
print('Done')

Done


# Three Panel Format for TEG Data by Hemorrhage Level

In [9]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------
# Parameters
# ------------------------
file_path = 'ERNE_TEG_cleaned.xlsx'
teg_metrics = ['R_time', 'K_time', 'Angle', 'MA', 'G', 'LY30']

AXIS_TICK_SIZE = 16
TITLE_SIZE = 20
LABEL_SIZE = 18

# Marker symbols and colors
marker_info = {
    "No": {"marker": "o", "color": "black"},
    "Partial": {"marker": "o", "color": "blue"},
    "Full": {"marker": "o", "color": "red"}
}
legend_order = ["No", "Partial", "Full"]  # ensure consistent legend order

# Output directory
output_dir = 'TEG_By_Hemorrhage_Level'
os.makedirs(output_dir, exist_ok=True)

# ------------------------
# Load and preprocess data
# ------------------------
df = pd.read_excel(file_path)

time_col = 'Time Point'
group_col = 'Occlusion Group'
level_col = 'Hemorrhage Level'

# Ensure numeric
df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
df = df.dropna(subset=[time_col])
df[time_col] = df[time_col].astype(float)

# Correct "None Occlusion" → "No Occlusion"
df[group_col] = df[group_col].replace({"None Occlusion": "No Occlusion"})

# Map group to simplified keys for marker/color
def map_group_key(name):
    if "No" in name:
        return "No"
    elif "Partial" in name:
        return "Partial"
    elif "Full" in name:
        return "Full"
    return "No"

df['Group Key'] = df[group_col].apply(map_group_key)

# ------------------------
# Plotting function
# ------------------------
def plot_teg_panels():
    hemorrhage_levels = sorted(df[level_col].unique())

    for metric in teg_metrics:
        fig, axes = plt.subplots(1, len(hemorrhage_levels), figsize=(18, 6), sharey=True)
        if len(hemorrhage_levels) == 1:
            axes = [axes]

        handles, labels = [], []

        for ax, hemorrhage_level in zip(axes, hemorrhage_levels):
            level_df = df[df[level_col] == hemorrhage_level]

            # Group by Occlusion Group & Time Point
            grouped_df = level_df.groupby(['Group Key', time_col])[metric].agg(['mean', 'std']).reset_index()

            for key in legend_order:  # enforce legend order
                subset = grouped_df[grouped_df['Group Key'] == key]
                if subset.empty:
                    continue

                label_name = f"{key} Occlusion"

                line, = ax.plot(
                    subset[time_col],
                    subset['mean'],
                    marker=marker_info[key]["marker"],
                    markersize=8,
                    linewidth=2,
                    color=marker_info[key]["color"],
                    label=label_name
                )
                if label_name not in labels:
                    handles.append(line)
                    labels.append(label_name)

                # Error bars
                ax.fill_between(
                    subset[time_col],
                    subset['mean'] - subset['std'],
                    subset['mean'] + subset['std'],
                    alpha=0.2,
                    color=marker_info[key]["color"]
                )

            # Reference line at y=1
            ax.axhline(y=1, color='gray', linestyle='--', linewidth=1)

            ax.set_title(f"{hemorrhage_level}% Hemorrhage - {metric}", fontsize=TITLE_SIZE)
            ax.set_xlabel("Time", fontsize=LABEL_SIZE)
            ax.grid(True, which='both', linestyle='--', linewidth=0.5)
            ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

        axes[0].set_ylabel(metric, fontsize=LABEL_SIZE)

        # Legend next to right-most panel
        axes[-1].legend(
            handles, labels,
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            fontsize=12,
            borderaxespad=0
        )

        plt.tight_layout()
        png_file = os.path.join(output_dir, f'TEG_{metric}_Panel.png')
        plt.savefig(png_file, dpi=300, bbox_inches='tight')
        plt.close()

# ------------------------
# Run plotting
# ------------------------
plot_teg_panels()


# And Now By Occlusion Group

In [11]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------
# Parameters
# ------------------------
file_path = 'ERNE_TEG_cleaned.xlsx'
teg_metrics = ['R_time', 'K_time', 'Angle', 'MA', 'G', 'LY30']

AXIS_TICK_SIZE = 16
TITLE_SIZE = 20
LABEL_SIZE = 18

# Marker symbols and colors — now for hemorrhage levels
marker_info = {
    "10": {"marker": "o", "color": "orange"},
    "20": {"marker": "o", "color": "purple"},
    "30": {"marker": "o", "color": "brown"},
}
# Keep order consistent (sorted ascending numerically)
legend_order = sorted(marker_info.keys(), key=lambda x: float(x))

# Output directory
output_dir = 'TEG_By_Occlusion_Group'
os.makedirs(output_dir, exist_ok=True)

# ------------------------
# Load and preprocess data
# ------------------------
df = pd.read_excel(file_path)

time_col = 'Time Point'
group_col = 'Occlusion Group'
level_col = 'Hemorrhage Level'

# Ensure numeric time
df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
df = df.dropna(subset=[time_col])
df[time_col] = df[time_col].astype(float)

# Normalize occlusion group names
df[group_col] = df[group_col].replace({
    "None Occlusion": "No Occlusion"
})

# Simplify occlusion group keys for naming/ordering
def map_group_key(name):
    if "No" in name:
        return "No"
    elif "Partial" in name:
        return "Partial"
    elif "Full" in name:
        return "Full"
    return "No"

df['Group Key'] = df[group_col].apply(map_group_key)

# Define consistent order for occlusion groups
occlusion_order = ["No", "Partial", "Full"]

# ------------------------
# Plotting function
# ------------------------
def plot_teg_panels():
    occlusion_groups = [g for g in occlusion_order if g in df['Group Key'].unique()]

    for metric in teg_metrics:
        fig, axes = plt.subplots(1, len(occlusion_groups), figsize=(18, 6), sharey=True)
        if len(occlusion_groups) == 1:
            axes = [axes]

        handles, labels = [], []

        for ax, occ_group in zip(axes, occlusion_groups):
            group_df = df[df['Group Key'] == occ_group]

            # Group by Hemorrhage Level & Time Point
            grouped_df = group_df.groupby([level_col, time_col])[metric].agg(['mean', 'std']).reset_index()

            for level_key in legend_order:
                subset = grouped_df[grouped_df[level_col].astype(str) == level_key]
                if subset.empty:
                    continue

                label_name = f"{level_key}% Hemorrhage"

                line, = ax.plot(
                    subset[time_col],
                    subset['mean'],
                    marker=marker_info[level_key]["marker"],
                    markersize=8,
                    linewidth=2,
                    color=marker_info[level_key]["color"],
                    label=label_name
                )

                if label_name not in labels:
                    handles.append(line)
                    labels.append(label_name)

                # Error shading
                ax.fill_between(
                    subset[time_col],
                    subset['mean'] - subset['std'],
                    subset['mean'] + subset['std'],
                    alpha=0.2,
                    color=marker_info[level_key]["color"]
                )

            ax.axhline(y=1, color='gray', linestyle='--', linewidth=1)
            ax.set_title(f"{occ_group} Occlusion - {metric}", fontsize=TITLE_SIZE)
            ax.set_xlabel("Time", fontsize=LABEL_SIZE)
            ax.grid(True, linestyle='--', linewidth=0.5)
            ax.tick_params(axis='both', labelsize=AXIS_TICK_SIZE)

        axes[0].set_ylabel(metric, fontsize=LABEL_SIZE)

        # Legend outside the right-most panel
        axes[-1].legend(
            handles, labels,
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            fontsize=12,
            borderaxespad=0
        )

        plt.tight_layout()
        png_file = os.path.join(output_dir, f'TEG_{metric}_Panel.png')
        plt.savefig(png_file, dpi=300, bbox_inches='tight')
        plt.close()

# ------------------------
# Run plotting
# ------------------------
plot_teg_panels()
